# FIM-LoRA Smoke Test

Validates the full pipeline end-to-end on a tiny setup before running the real experiments.

**What this tests:**
1. FIM accumulation — does it collect squared gradients correctly?
2. Rank allocation — does it assign higher rank to more sensitive layers?
3. Adapter resizing — do lora_A/lora_B shapes update correctly?
4. Training loop — does a model fine-tune without errors after FIM reallocation?
5. Baseline comparison — FIM-LoRA vs standard LoRA on a tiny dataset

**Runtime:** ~5 min on CPU, ~1 min on GPU  
**No SageMaker needed** — runs entirely locally

## 0. Setup

In [ ]:
# HuggingFace token — loaded automatically from ~/.huggingface/token
# If that file does not exist, set HF_TOKEN env var or paste token below.
# Run scripts/setup_hf_token.sh once to persist the token permanently.
import os
from pathlib import Path

# Priority: env var > ~/.huggingface/token
hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
if not hf_token:
    token_file = Path.home() / ".huggingface" / "token"
    if token_file.exists():
        hf_token = token_file.read_text().strip()

if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token
    print(f"HF token loaded ✓ (starts with: {hf_token[:8]}...)")
else:
    print("WARNING: No HF token found.")
    print("  Run: bash scripts/setup_hf_token.sh")
    print("  Or:  os.environ["HF_TOKEN"] = "hf_your_token"")  # paste here


In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

import warnings
warnings.filterwarnings('ignore')

import torch
import numpy as np
from pprint import pprint

print(f"PyTorch: {torch.__version__}")
print(f"Device:  {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

import peft
import transformers
print(f"PEFT:           {peft.__version__}")
print(f"Transformers:   {transformers.__version__}")

## 1. Build a tiny model + LoRA

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from peft import LoraConfig, get_peft_model

# Use a small BERT variant so this runs on CPU in seconds
MODEL = "FacebookAI/roberta-base"  # 125M params — same architecture as DeBERTa used in real experiments

tokenizer = AutoTokenizer.from_pretrained(MODEL)
base_model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)

config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["query", "value", "key"],
    bias="none",
    task_type="SEQ_CLS",
)

model = get_peft_model(base_model, config)
model.print_trainable_parameters()
print(f"\nTarget modules: {config.target_modules}")

## 2. Build a tiny calibration dataloader

In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset

class TinyDataset(Dataset):
    """16 random classification examples."""
    def __init__(self, tokenizer, n=16, max_len=64):
        sentences = [
            "The model failed to converge during training.",
            "Gradient descent is an optimization algorithm.",
            "LoRA reduces trainable parameters significantly.",
            "Fisher Information measures parameter sensitivity.",
            "Attention layers are critical for language models.",
            "Fine-tuning adapts pretrained weights to new tasks.",
            "Rank allocation affects model expressiveness.",
            "Empirical FIM approximates second-order information.",
            "Low-rank adaptation is parameter efficient.",
            "Calibration batches estimate gradient statistics.",
            "GLUE benchmark tests natural language understanding.",
            "DeBERTa achieves strong NLU performance.",
            "Squared gradients accumulate FIM diagonal estimates.",
            "Budget constraints preserve mean rank across layers.",
            "Adapter resizing modifies lora_A and lora_B shapes.",
            "Importance scores guide non-uniform rank allocation.",
        ]
        self.encodings = tokenizer(
            sentences, truncation=True, max_length=max_len,
            padding="max_length", return_tensors="pt"
        )
        self.labels = torch.randint(0, 2, (n,))

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids":      self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "labels":         self.labels[idx],
        }

dataset = TinyDataset(tokenizer)
calib_loader = DataLoader(dataset, batch_size=4, shuffle=True)
print(f"Calibration batches: {len(calib_loader)}")
print(f"Batch keys: {list(next(iter(calib_loader)).keys())}")

## 3. Test FIM accumulation

In [ ]:
from fim_allocator import accumulate_fim

fim_scores = accumulate_fim(
    model=model,
    dataloader=calib_loader,
    n_batches=4,
    adapter_name="default",
)

print(f"Layers with FIM scores: {len(fim_scores)}")
print("\nPer-layer eFIM stats:")
print(f"  {'Layer':<50} {'Mean eFIM':>12} {'Max eFIM':>12} {'Shape'}")
print("-" * 90)
for name, fim in sorted(fim_scores.items()):
    print(f"  {name:<50} {fim.mean().item():>12.6f} {fim.max().item():>12.6f} {tuple(fim.shape)}")
if len(fim_scores) == 0:
    print("WARNING: No FIM scores captured — gradients did not flow through LoRA layers.")
    print("Check: model.train() called? Loss computed? LoRA layers have requires_grad=True?")
else:
    assert all(v.abs().sum() > 0 for v in fim_scores.values()), "All FIM scores are zero!"
    print("
FIM scores look healthy (non-zero) ✓")


## 4. Test importance scoring

In [ ]:
from fim_allocator import compute_importance
import matplotlib.pyplot as plt

importance = compute_importance(fim_scores)

# Sort by importance
sorted_imp = sorted(importance.items(), key=lambda x: -x[1])

print("Layer importance ranking (highest → lowest loss sensitivity):")
print(f"  {'Rank':<5} {'Layer':<50} {'Importance Score':>18}")
print("-" * 76)
for i, (name, score) in enumerate(sorted_imp, 1):
    max_score = sorted_imp[0][1] if sorted_imp[0][1] > 1e-12 else 1e-10
    bar = "█" * int(score / max_score * 20)
    print(f"  {i:<5} {name:<50} {score:>12.6f}  {bar}")

# Quick bar chart
fig, ax = plt.subplots(figsize=(10, 3))
names  = [n.split('.')[-3] + '.' + n.split('.')[-1] for n, _ in sorted_imp]
scores = [s for _, s in sorted_imp]
ax.barh(names[::-1], scores[::-1], color='steelblue')
ax.set_xlabel('Mean eFIM score (importance)')
ax.set_title('Per-layer FIM importance scores')
plt.tight_layout()
plt.savefig('../results/importance_scores.png', dpi=120, bbox_inches='tight')
plt.show()
print("Saved → results/importance_scores.png")

## 5. Test rank allocation

In [ ]:
from fim_allocator import allocate_ranks

base_r = 8
rank_pattern = allocate_ranks(importance, base_r=base_r, r_min=1, r_max=16)

print(f"Base rank (budget per layer): r={base_r}")
print(f"Total budget:                 {base_r * len(rank_pattern)} = {len(rank_pattern)} layers × r={base_r}")
print(f"Allocated total:              {sum(rank_pattern.values())}")
print()
print("Allocated ranks:")
for name, r in sorted(rank_pattern.items(), key=lambda x: -x[1]):
    marker = '▲' if r > base_r else ('▼' if r < base_r else '=')
    bar = '█' * r
    imp = importance.get(name, 0)
    print(f"  {marker} r={r:<3} | imp={imp:.4f} | {name:<50} {bar}")

# Verify budget
assert sum(rank_pattern.values()) == base_r * len(rank_pattern), "Budget violated!"
print(f"\n✓ Budget constraint satisfied: sum(ranks) = {sum(rank_pattern.values())} = {len(rank_pattern)} × {base_r}")

## 6. Test adapter resizing

In [ ]:
from fim_allocator import resize_lora_layer
from peft.tuners.lora.layer import Linear as LoraLinear

print("Before resizing:")
for name, module in model.named_modules():
    if isinstance(module, LoraLinear) and 'default' in module.lora_A:
        A = module.lora_A['default'].weight
        B = module.lora_B['default'].weight
        print(f"  {name:<50}  lora_A: {tuple(A.shape)}  lora_B: {tuple(B.shape)}  r={module.r['default']}")

print()

# Apply resizing
for name, module in model.named_modules():
    if isinstance(module, LoraLinear) and name in rank_pattern:
        resize_lora_layer(module, 'default', rank_pattern[name], adjust_scaling=True)

print("After resizing:")
for name, module in model.named_modules():
    if isinstance(module, LoraLinear) and 'default' in module.lora_A:
        A = module.lora_A['default'].weight
        B = module.lora_B['default'].weight
        expected_r = rank_pattern.get(name, base_r)
        status = '✓' if A.shape[0] == expected_r else '✗'
        print(f"  {status} {name:<50}  lora_A: {tuple(A.shape)}  lora_B: {tuple(B.shape)}  r={module.r['default']}")

print("\n✓ All adapter layers resized correctly")

## 7. Test full apply_fim_ranks pipeline

In [ ]:
from fim_allocator import apply_fim_ranks

# Fresh model for clean test
base_model2 = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)
model2 = get_peft_model(base_model2, LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.1,
    target_modules=["query", "value", "key"], bias="none", task_type="SEQ_CLS",
))

print("Running apply_fim_ranks (full pipeline)...\n")
rank_pattern2 = apply_fim_ranks(
    model=model2,
    dataloader=calib_loader,
    n_batches=4,
    r_min=1,
    r_max=16,
    verbose=True,
)

print(f"\nrank_pattern saved to config: {model2.peft_config['default'].rank_pattern}")

## 8. Mini training loop — FIM-LoRA vs standard LoRA

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import LinearLR

def train_loop(model, dataloader, n_steps=20, lr=2e-4, label=''):
    """Tiny training loop. Returns final loss."""
    model.train()
    optimizer = AdamW(model.parameters(), lr=lr)
    losses = []
    for step, batch in enumerate(dataloader):
        if step >= n_steps:
            break
        optimizer.zero_grad()
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
        if step % 5 == 0:
            print(f"  [{label}] step {step:>2}/{n_steps}  loss={loss.item():.4f}")
    return losses

# Repeat dataloader for more steps
from itertools import cycle
train_loader = DataLoader(dataset, batch_size=4, shuffle=True)
inf_loader = cycle(train_loader)

# --- Standard LoRA baseline ---
base_lora = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)
model_lora = get_peft_model(base_lora, LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.0,
    target_modules=["query", "value", "key"], bias="none", task_type="SEQ_CLS",
))
print("Training standard LoRA (r=8, fixed)...")
losses_lora = train_loop(model_lora, inf_loader, n_steps=20, label='LoRA')

print()

# --- FIM-LoRA ---
base_fim = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)
model_fim = get_peft_model(base_fim, LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.0,
    target_modules=["query", "value", "key"], bias="none", task_type="SEQ_CLS",
))
apply_fim_ranks(model_fim, calib_loader, n_batches=4, verbose=False)
print("Training FIM-LoRA (r=8 budget, FIM-allocated)...")
losses_fim = train_loop(model_fim, cycle(train_loader), n_steps=20, label='FIM-LoRA')

## 9. Plot training curves

In [ ]:
import matplotlib.pyplot as plt
import os
os.makedirs('../results', exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Training curves
ax = axes[0]
ax.plot(losses_lora, label='LoRA (fixed r=8)', color='gray', linewidth=2)
ax.plot(losses_fim,  label='FIM-LoRA (budget=8)', color='steelblue', linewidth=2)
ax.set_xlabel('Step')
ax.set_ylabel('Loss')
ax.set_title('Training Loss: LoRA vs FIM-LoRA\n(tiny smoke test — not paper results)')
ax.legend()
ax.grid(alpha=0.3)

# Rank distribution
ax2 = axes[1]
from peft.tuners.lora.layer import Linear as LoraLinear
layer_names, fim_ranks, lora_ranks = [], [], []
for name, module in model_fim.named_modules():
    if isinstance(module, LoraLinear) and 'default' in module.lora_A:
        short = name.split('.')[-3] + '.' + name.split('.')[-1]
        layer_names.append(short)
        fim_ranks.append(module.r['default'])
        lora_ranks.append(8)  # fixed

x = range(len(layer_names))
ax2.bar([i - 0.2 for i in x], lora_ranks, 0.4, label='LoRA (fixed)', color='gray', alpha=0.7)
ax2.bar([i + 0.2 for i in x], fim_ranks,  0.4, label='FIM-LoRA',     color='steelblue', alpha=0.9)
ax2.axhline(8, color='gray', linestyle='--', linewidth=0.8, label='Base r=8')
ax2.set_xticks(list(x))
ax2.set_xticklabels(layer_names, rotation=45, ha='right', fontsize=8)
ax2.set_ylabel('Rank')
ax2.set_title('Rank allocation: LoRA vs FIM-LoRA')
ax2.legend()
ax2.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../results/smoke_test_results.png', dpi=120, bbox_inches='tight')
plt.show()
print("Saved → results/smoke_test_results.png")

## 10. Summary

In [ ]:
print("=" * 55)
print("  FIM-LoRA Smoke Test — Summary")
print("=" * 55)

checks = [
    ("FIM accumulation",          len(fim_scores) > 0),
    ("Importance scores computed", len(importance) == len(fim_scores)),
    ("Budget constraint satisfied",sum(rank_pattern.values()) == base_r * len(rank_pattern)),
    ("Adapter layers resized",     all(m.r['default'] == rank_pattern.get(n, base_r)
                                       for n, m in model2.named_modules()
                                       if isinstance(m, LoraLinear) and n in rank_pattern)),
    ("FIM-LoRA training completes",len(losses_fim) == 20),
    ("LoRA baseline trains",       len(losses_lora) == 20),
]

all_pass = True
for label, ok in checks:
    status = '✓' if ok else '✗'
    print(f"  {status}  {label}")
    if not ok:
        all_pass = False

print()
print(f"  Final loss  — LoRA:     {losses_lora[-1]:.4f}")
print(f"  Final loss  — FIM-LoRA: {losses_fim[-1]:.4f}")
print(f"  FIM-allocated ranks:    {dict(sorted(rank_pattern.items()))}")
print()
if all_pass:
    print("  ✅ All checks passed — ready to run SageMaker experiments")
    print("     → bash scripts/run_glue_sweep.sh")
else:
    print("  ❌ Some checks failed — review output above")